# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1
The paper reports that growing content tends to be longer, younger, and slightly better positioned than declining content.

**Methodology question**

A question I would ask is how the labels for "growing" and "declining" were created. The paper explains that trend direction comes from changes in impressions over consecutive 30-day periods. I would also ask whether any of those same measurements were later used as model features, because that could introduce label leakage.

The paper clearly states that these comparisons are observational rather than causal, which is an appropriate limitation.

---

### Finding 2
The paper reports that refreshed pages older than one year perform much better than similarly old pages that were not refreshed.

**Methodology question**

I would ask whether the comparison controls for differences between pages before they were refreshed. Pages chosen for refresh may already have been higher-quality or more valuable pages, so part of the improvement may come from selection rather than the refresh itself.

I also appreciate that the paper avoids claiming causation and describes this finding as an observed relationship rather than proof that refreshing alone causes the improvement.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)


# Create target label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


features = [
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]


X = df[features].copy()

y = df["is_declining_label"]

groups = df["client_id"]


print("Features:", X.shape)
print("Label distribution:")
print(y.value_counts())

Features: (30000, 10)
Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I evaluated my Logistic Regression model using a random train/test split.

For this validation audit, I re-ran the model using a grouped split based on `client_id`. This prevents pages from the same client from appearing in both the training and testing sets, making the evaluation more representative of how the model would perform on unseen clients.

| Validation Method | Precision |
|-------------------|----------:|
| Random split (Week 5) | 0.58 |
| Grouped split (Week 6) | 0.621 |

The grouped split produced a precision of 0.621 after removing rows with missing feature values. This is slightly higher than the random split result of 0.58. In this dataset, grouping by client did not reduce the model's performance. However, because the grouped evaluation uses unseen clients and a cleaned dataset, it should be considered the more reliable estimate of model performance.

The grouped evaluation should therefore be considered the more reliable estimate of model performance.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

# Same features used in Week 5
feature_cols = [
    "days_since_last_update",
    "word_count",
    "impressions_90d",
    "search_volume",
    "avg_position"
]

# Keep only rows with complete feature values
data = df[feature_cols + ["is_declining_label", "client_id"]].dropna()

X = data[feature_cols]
y = data["is_declining_label"]
groups = data["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

precision = precision_score(y_test, pred)

print("Rows after removing missing values:", len(data))
print("Grouped Precision:", round(precision, 3))

Rows after removing missing values: 20018
Grouped Precision: 0.621


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I reviewed each feature used in my final Logistic Regression model to identify potential sources of label leakage or information that would not be available at prediction time.

| Feature | Leakage Risk | Reason |
|----------|--------------|--------|
| days_since_last_update | Low | Available before prediction time. |
| word_count | Low | Static page metadata. Rows with missing values were removed before training. |
| impressions_90d | Medium | Represents historical performance and is appropriate only if measured before the prediction window. |
| search_volume | Low | External search demand estimate available before prediction. |
| avg_position | Medium | Safe only when measured before the prediction period. |

I deliberately excluded the following columns from the model:

- `trend_direction`
- `trend_pct`

These columns are directly related to the target label and would introduce label leakage if used as model features.

I also excluded identifiers such as `content_id` and `client_id` from the feature set. The `client_id` column was used only for the grouped train/test split and was not used for model training.

Finally, before training the model, I removed rows with missing feature values to ensure that Logistic Regression received complete input data.
Based on this review, I did not find evidence that my final feature set directly leaks the target label into the model.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Features used:")
print(feature_cols)

Features used:
['days_since_last_update', 'word_count', 'impressions_90d', 'search_volume', 'avg_position']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

"My Logistic Regression model accurately predicts declining content."

### Revised claim

Using the selected features, the Logistic Regression model achieved a measured precision of 0.58 using a random split and 0.621 using a grouped client split. These observed results suggest that the model may help prioritize pages for review as a decision-support tool. The results are directional and should not be interpreted as proof that the model will generalize to every client or predict future performance with certainty.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.